<a href="https://colab.research.google.com/github/zombimann/Mathematical-video-animations-and-visualization/blob/main/Spring_Mass_Damper_Impulse_Response.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Spring-Mass-Damper System Simulation

This notebook provides a high-fidelity numerical simulation and visualization of a second-order mechanical system. The project demonstrates the behavior of a mass-spring-damper assembly under different damping regimes, specifically focusing on the system response to an impulsive force.

## Theoretical Background

The motion of a spring-mass-damper system is governed by a linear second-order differential equation derived from Newton's Second Law:

$m \frac{d^2x}{dt^2} + c \frac{dx}{dt} + kx = F(t)$

Where:
* **m** is the mass
* **c** is the damping coefficient
* **k** is the spring constant
* **x** is the displacement from equilibrium

The system's behavior is categorized by the damping ratio (zeta), defined as $\zeta = \frac{c}{2\sqrt{mk}}$.

### Damping Regimes

1. **Underdamped ($ζ < 1$):** The system oscillates with an exponentially decaying amplitude. The energy dissipates slowly, allowing for multiple cycles before returning to equilibrium.
2. **Critically Damped ($ζ = 1$):** The system returns to equilibrium as quickly as possible without oscillating. This is often the design goal for automotive suspensions and door closers.
3. **Overdamped ($ζ > 1$):** The system returns to equilibrium without oscillation but more slowly than in the critically damped case due to high resistive forces.

## Simulation Features

* **Regime Transitions:** The simulation cycles through all three damping states in a single continuous animation.
* **Impulse Logic:** Each regime is initiated by a discrete impulse, represented visually by a directional arrow.
* **Real-time Plotting:** A dynamic trace shows the displacement $x(t)$ synchronized with the mechanical animation.
* **Visual Fidelity:** The animation includes a vertically extruded mass, functional spring coils, and a piston-style damper mechanism.

In [3]:
!sudo apt-get update && sudo apt-get install -y libcairo2-dev libpango1.0-dev ffmpeg texlive texlive-latex-extra texlive-fonts-extra texlive-latex-recommended texlive-science texlive-fonts-recommended dvisvgm -q
!pip install manim -q


Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 1s (3,541 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists...
Building dependency tree...
Reading state information...
libcairo2-dev is already the newest version (1.16.0-5ubuntu2.1).


In [4]:
# Force path update awareness
import os
os.environ['PATH'] += ':/usr/bin:/usr/local/bin'

In [22]:
"""
Author: Mugambi Ndwiga
Instagram: @craftsandengineering
Level: High School
Concept: Spring-Mass-Damper System – Three Damping Regimes
GitHub: https://github.com/zombimann/Mathematical-video-animations-and-visualization
"""

import os
import tempfile
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.patches import FancyBboxPatch, Rectangle, Circle
from matplotlib import rcParams

# =========================
# PARAMETRIC SETTINGS
# =========================
mass = 1.0
spring_k = 10.0

c_under = 0.5
c_crit = 2.0 * np.sqrt(mass * spring_k)
c_over = 20.0

zeta_under = c_under / (2 * np.sqrt(mass * spring_k))
zeta_crit = 1.0
zeta_over = c_over / (2 * np.sqrt(mass * spring_k))

impulse_velocity = 4.6

title_dur = 2.0
phase_durations = {
    "under": 5.8,
    "critical": 3.6,
    "over": 6.0,
}
transition_dur = 1.2
closing_fade_dur = 0.40
closing_hold_dur = 1.50

fps = 20
phase_order = ["under", "critical", "over"]

fig_w, fig_h = 12.8, 7.2
dpi = 100

COLOR_BG = "#F4F6F8"
COLOR_PANEL = "#FFFFFF"
COLOR_PANEL_BORDER = "#D0D7DE"
COLOR_CLOSE_BG = "#1A237E"
COLOR_MASS = "#1976D2"
COLOR_SPRING = "#F57C00"
COLOR_DAMPER = "#616161"
COLOR_WALL = "#424242"
COLOR_GROUND = "#424242"
COLOR_ACCENT = "#1E88E5"
COLOR_TEXT = "#1F1F1F"
COLOR_SUBTLE = "#5F6368"
TRACE_COLORS = {
    "under": "#1E88E5",
    "critical": "#43A047",
    "over": "#E53935",
}

rcParams["font.family"] = "sans-serif"
rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial", "Helvetica"]
rcParams["font.size"] = 13

wn = np.sqrt(spring_k / mass)

def impulse_response(t, c, v0):
    zeta = c / (2 * np.sqrt(mass * spring_k))
    if zeta < 1:
        wd = wn * np.sqrt(1.0 - zeta**2)
        return (v0 / wd) * np.exp(-zeta * wn * t) * np.sin(wd * t)
    if np.isclose(zeta, 1.0):
        return v0 * t * np.exp(-wn * t)
    s1 = -zeta * wn + wn * np.sqrt(zeta**2 - 1.0)
    s2 = -zeta * wn - wn * np.sqrt(zeta**2 - 1.0)
    return (v0 / (s1 - s2)) * (np.exp(s1 * t) - np.exp(s2 * t))

def make_spring_path(x_start, x_end, y_center, coils=7, amplitude=0.17):
    if x_end <= x_start + 0.01:
        return np.array([x_start, x_end]), np.array([y_center, y_center])
    t = np.linspace(0, 2 * np.pi * coils, 220)
    x = x_start + (x_end - x_start) * t / (2 * np.pi * coils)
    y = y_center + amplitude * np.sin(t)
    return x, y

total_duration = title_dur + sum(phase_durations.values()) + (len(phase_order)-1) * transition_dur + closing_fade_dur + closing_hold_dur
total_frames = int(np.ceil(total_duration * fps))

phase_info = {
    "under": {"title": "Underdamped", "subtitle": f"\u03b6 = {zeta_under:.2f} < 1", "color": TRACE_COLORS["under"], "c": c_under},
    "critical": {"title": "Critically damped", "subtitle": "\u03b6 = 1.00", "color": TRACE_COLORS["critical"], "c": c_crit},
    "over": {"title": "Overdamped", "subtitle": f"\u03b6 = {zeta_over:.2f} > 1", "color": TRACE_COLORS["over"], "c": c_over},
}

fig = plt.figure(figsize=(fig_w, fig_h), dpi=dpi, facecolor=COLOR_BG)
ax_main = fig.add_axes([0.03, 0.05, 0.62, 0.90], facecolor=COLOR_BG)
ax_main.set_xlim(-4.3, 4.9)
ax_main.set_ylim(-2.35, 3.45)
ax_main.set_aspect("equal")
ax_main.axis("off")

def make_card_axes(bounds):
    ax = fig.add_axes(bounds, facecolor=COLOR_PANEL)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")
    card = FancyBboxPatch((0, 0), 1, 1, boxstyle="round,pad=0.04,rounding_size=0.04", fc=COLOR_PANEL, ec=COLOR_PANEL_BORDER, lw=1.5)
    ax.add_patch(card)
    return ax

ax_info = make_card_axes([0.67, 0.60, 0.30, 0.31])
ax_plot_card = make_card_axes([0.67, 0.12, 0.30, 0.42])

title_text = fig.text(0.50, 0.965, "Spring-Mass-Damper System", ha="center", va="top", fontsize=23, fontweight="bold", color=COLOR_TEXT)
subtitle_text = fig.text(0.50, 0.915, "Response to impulsive force across three regimes", ha="center", va="top", fontsize=12.5, color=COLOR_SUBTLE)

# Professional Math Expressions in center-top
math_ode = fig.text(0.43, 0.865, r"$m \cdot x'' + c \cdot x' + k \cdot x = F(t)$", ha="center", va="top", fontsize=16, color=COLOR_TEXT)
math_zeta = fig.text(0.43, 0.815, r"$\zeta = \frac{c}{2\sqrt{mk}}$", ha="center", va="top", fontsize=16, color=COLOR_TEXT)

# Dynamic labels for mechanical elements
label_k = ax_main.text(-1.7, 0.85, "k", fontsize=14, fontweight="bold", color=COLOR_SPRING, ha="center")
label_c = ax_main.text(-1.7, -0.35, "c", fontsize=14, fontweight="bold", color=COLOR_DAMPER, ha="center")
label_m = ax_main.text(0.0, -1.2, "m", fontsize=14, fontweight="bold", color="white", ha="center", zorder=10)

phase_badge = ax_main.text(-4.05, 3.50, "", ha="left", va="center", fontsize=17, fontweight="bold", color=COLOR_TEXT, bbox=dict(boxstyle="round,pad=0.35,rounding_size=0.18", fc="white", ec=COLOR_PANEL_BORDER, lw=1.0), alpha=0.0)
impulse_arrow = ax_main.annotate("", xy=(0, 0), xytext=(-0.8, 0), arrowprops=dict(arrowstyle="->,head_width=0.5,head_length=0.7", lw=4.5, color=COLOR_SPRING), alpha=0.0, visible=False)

ground_y = -1.55
wall_x = -3.45
mass_width = 1.25
mass_height = 1.85
mass_rest_x = 0.0

ax_main.plot([-4.2, 4.7], [ground_y, ground_y], color=COLOR_GROUND, lw=6, solid_capstyle="round")
wall = Rectangle((wall_x - 0.12, -1.68), 0.24, 3.35, fc=COLOR_WALL, ec=COLOR_WALL, lw=0, zorder=3)
ax_main.add_patch(wall)

mass_rect = Rectangle((mass_rest_x - mass_width / 2, -1.0), mass_width, mass_height, fc=COLOR_MASS, ec="#0D47A1", lw=2.0, zorder=6)
ax_main.add_patch(mass_rect)

wheel_radius = 0.15
wheel_left = Circle((mass_rest_x - 0.34, ground_y + wheel_radius), wheel_radius, fc="#222222", ec="none", zorder=7)
wheel_right = Circle((mass_rest_x + 0.34, ground_y + wheel_radius), wheel_radius, fc="#222222", ec="none", zorder=7)
ax_main.add_patch(wheel_left)
ax_main.add_patch(wheel_right)

spring_line, = ax_main.plot([], [], color=COLOR_SPRING, lw=3.2, zorder=4)
damper_body = Rectangle((wall_x + 0.05, -0.65), 0.85, 0.18, fc=COLOR_DAMPER, ec="none", zorder=4)
damper_piston = Rectangle((0, -0.70), 0.16, 0.28, fc=COLOR_DAMPER, ec="none", zorder=4)
damper_rod, = ax_main.plot([], [], color=COLOR_DAMPER, lw=3.0, zorder=4)
ax_main.add_patch(damper_body)
ax_main.add_patch(damper_piston)

disp_arrow = ax_main.annotate("", xy=(mass_rest_x, 1.25), xytext=(mass_rest_x, 0.95), arrowprops=dict(arrowstyle="->", lw=2.2, color=COLOR_TEXT))
disp_label = ax_main.text(mass_rest_x, 1.35, "x(t)", ha="center", va="bottom", fontsize=14.5, color=COLOR_TEXT)

info_regime = ax_info.text(0.50, 0.54, "", ha="center", va="center", fontsize=19, fontweight="bold", color=COLOR_TEXT)
info_zeta = ax_info.text(0.50, 0.28, "", ha="center", va="center", fontsize=14, color=COLOR_SUBTLE)

ax_plot = fig.add_axes([0.70, 0.17, 0.24, 0.20], facecolor="none")
ax_plot.spines["top"].set_visible(False)
ax_plot.spines["right"].set_visible(False)
ax_plot.set_xlabel("Time (s)", fontsize=10, labelpad=2)
ax_plot.set_ylabel("x (m)", fontsize=10, labelpad=2)
ax_plot.set_xlim(0, 6.5)
ax_plot.set_ylim(-1.5, 1.65)
ax_plot.grid(True, alpha=0.15)

plot_line, = ax_plot.plot([], [], lw=2.8)
plot_dot, = ax_plot.plot([], [], "o", ms=6.5)

ax_main.text(4.8, -2.2, "Mugambi Ndwiga | @craftsandengineering", ha="right", va="bottom", fontsize=11, color=COLOR_SUBTLE, alpha=0.45, fontweight="bold")

ax_overlay = fig.add_axes([0, 0, 1, 1], zorder=150)
ax_overlay.axis("off")
overlay_patch = Rectangle((0, 0), 1, 1, transform=ax_overlay.transAxes, fc="white", ec="none", alpha=0.0)
overlay_text = ax_overlay.text(0.5, 0.5, "", ha="center", va="center", fontsize=24, fontweight="bold", color=COLOR_TEXT, alpha=0.0)

ax_close = fig.add_axes([0, 0, 1, 1], zorder=200)
ax_close.axis("off")
close_bg = Rectangle((0, 0), 1, 1, transform=ax_close.transAxes, fc=COLOR_CLOSE_BG, alpha=0.0)
close_text = ax_close.text(0.5, 0.54, "Made by Mugambi Ndwiga\n@craftsandengineering", ha="center", va="center", fontsize=28, color="white", fontweight="bold", alpha=0.0)
ax_close.add_patch(close_bg)

def current_phase_state(t):
    cursor = 0.0
    if t < title_dur: return ("title", None, t, None)
    cursor = title_dur
    for i, phase in enumerate(phase_order):
        dur = phase_durations[phase]
        if t < cursor + dur: return ("phase", phase, t - cursor, None)
        cursor += dur
        if i < len(phase_order) - 1:
            if t < cursor + transition_dur: return ("transition", phase_order[i+1], t - cursor, phase)
            cursor += transition_dur
    if t < cursor + closing_fade_dur: return ("fadeout", None, t - cursor, None)
    return ("closing", None, t - cursor - closing_fade_dur, None)

def update_visuals(phase, local_t, impulse_active=True):
    disp = impulse_response(local_t, phase_info[phase]["c"], impulse_velocity)
    color = phase_info[phase]["color"]
    mass_x = mass_rest_x + disp
    mass_rect.set_x(mass_x - mass_width/2)
    label_m.set_position((mass_x, -0.15))
    wheel_left.center = (mass_x - 0.34, ground_y + wheel_radius)
    wheel_right.center = (mass_x + 0.34, ground_y + wheel_radius)
    sx, sy = make_spring_path(wall_x + 0.12, mass_x - mass_width/2, 0.55)
    spring_line.set_data(sx, sy)
    damper_rod.set_data([wall_x + 0.9, mass_x - mass_width/2], [-0.56, -0.56])
    damper_piston.set_x(mass_x - mass_width/2 - 0.08)
    disp_arrow.xy = (mass_x, 1.20)
    disp_arrow.set_position((mass_x, 1.30))
    disp_label.set_position((mass_x, 1.40))
    phase_badge.set_text(phase_info[phase]["title"])
    phase_badge.set_alpha(1.0)
    info_regime.set_text(phase_info[phase]["title"])
    info_regime.set_color(color)
    info_zeta.set_text(phase_info[phase]["subtitle"])
    t_hist = np.linspace(0, local_t, int(local_t*fps)+1)
    x_hist = [impulse_response(ti, phase_info[phase]["c"], impulse_velocity) for ti in t_hist]
    plot_line.set_data(t_hist, x_hist)
    plot_line.set_color(color)
    plot_dot.set_data([t_hist[-1]], [x_hist[-1]])
    plot_dot.set_color(color)
    if impulse_active and local_t < 0.4:
        impulse_arrow.set_visible(True)
        impulse_arrow.set_alpha(max(0.0, 1.0 - local_t / 0.4))
        impulse_arrow.xy = (mass_x - mass_width/2, -0.1)
        impulse_arrow.set_position((mass_x - mass_width/2 - 0.7, -0.1))
    else:
        impulse_arrow.set_visible(False)
        impulse_arrow.set_alpha(0.0)

def update(frame):
    t = frame / fps
    kind, phase, local_t, prev = current_phase_state(t)
    overlay_patch.set_alpha(0.0)
    overlay_text.set_alpha(0.0)
    if kind == "title":
        update_visuals("under", 0.0, False)
        phase_badge.set_alpha(0.0)
        info_regime.set_text("")
        info_zeta.set_text("")
        intro_tri = 1.0 - max(0.0, min(1.0, local_t / (title_dur * 0.5)))
        overlay_patch.set_alpha(intro_tri)
        overlay_text.set_text("Spring-Mass-Damper\nSimulation")
        overlay_text.set_alpha(1.0 - intro_tri)
    elif kind == "phase":
        update_visuals(phase, local_t)
    elif kind == "transition":
        u = local_t / transition_dur
        tri = max(0.0, min(1.0, 1.0 - abs(2.0 * u - 1.0)))
        if u < 0.5:
            update_visuals(prev, phase_durations[prev] + local_t, False)
        else:
            update_visuals(phase, local_t - transition_dur/2, True)
        overlay_patch.set_alpha(tri * 0.9)
        overlay_text.set_text(f"Entering {phase_info[phase]['title']}")
        overlay_text.set_alpha(tri)
    elif kind == "fadeout":
        u = max(0.0, min(1.0, local_t / closing_fade_dur))
        overlay_patch.set_alpha(u)
    elif kind == "closing":
        u = max(0.0, min(1.0, local_t / 0.5))
        close_bg.set_alpha(u)
        close_text.set_alpha(u)
    return (mass_rect, spring_line, damper_rod, plot_line, plot_dot, overlay_patch, close_bg, impulse_arrow, label_k, label_c, label_m)

ani = animation.FuncAnimation(fig, update, frames=total_frames, interval=1000/fps, blit=False)
out_path = "./edited_spring_mass_damper.mp4"
ani.save(out_path, writer=animation.FFMpegWriter(fps=fps, bitrate=1500))
plt.close(fig)
print(f"Saved: {out_path}")

Saved: ./edited_spring_mass_damper.mp4


In [23]:
from IPython.display import Video
from google.colab import files

# Display the video
display(Video('./edited_spring_mass_damper.mp4', embed=True, width=800))

# Trigger download
files.download('./edited_spring_mass_damper.mp4')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>